In [1]:
import os
from strands.models.gemini import GeminiModel
from dotenv import load_dotenv

load_dotenv()

MODEL_ID = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")

model = GeminiModel(
    client_args={
        "vertexai": True,
    },
    model_id=MODEL_ID,
)

print(f"Using {MODEL_ID} on Gemini")

Using gemini-3.5-flash on Gemini


In [2]:
from strands import Agent
from strands.multiagent import GraphBuilder

In [3]:
risk_analyzer = Agent(
    name="risk_analyzer",
    model=model,
    callback_handler=None,
    system_prompt="""Analyze the order's risk level and pick exactly ONE category:
- HIGH_VALUE: orders over $1,000
- SUSPICIOUS: mismatched billing/shipping country, brand-new account with a large order, or known fraud patterns
- LOW_RISK: regular orders from trusted customers
If more than one applies, SUSPICIOUS wins over HIGH_VALUE.
Give a one-paragraph explanation, then finish with a final line in exactly this format:
RISK_CATEGORY: <HIGH_VALUE|SUSPICIOUS|LOW_RISK>""",
)

manual_review_agent = Agent(
    name="manual_review_agent",
    model=model,
    callback_handler=None,
    system_prompt="Review high-value orders. Verify customer identity, check order history, and approve or flag for further review.",
)

fraud_detection_agent = Agent(
    name="fraud_detection_agent",
    model=model,
    callback_handler=None,
    system_prompt="Run fraud detection on suspicious orders: known fraud patterns, velocity checks, and device fingerprinting. Decide whether to proceed.",
)

inventory_agent = Agent(
    name="inventory_agent",
    model=model,
    callback_handler=None,
    system_prompt="""Check inventory for the order items. Assume laptops, books and phones are in stock and
anything described as 'limited edition' or 'pre-order' is out of stock.
Explain briefly, then finish with a final line in exactly this format:
STOCK_STATUS: <IN_STOCK|OUT_OF_STOCK>""",
)

shipping_agent = Agent(
    name="shipping_agent",
    model=model,
    callback_handler=None,
    system_prompt="Generate a shipping label and process the shipment. Confirm a tracking number and estimated delivery date.",
)

refund_agent = Agent(
    name="refund_agent",
    model=model,
    callback_handler=None,
    system_prompt="Process a refund for out-of-stock items. Notify the customer and initiate the payment reversal.",
)

print("✓ All agents created")

✓ All agents created


In [4]:
import re


def decision(state, node_id: str, label: str) -> str:
    """Return the value of the final 'LABEL: VALUE' line written by a node."""
    text = str(state.results.get(node_id, ""))
    matches = re.findall(rf"{label}\s*:\s*([A-Z_]+)", text.upper())
    return matches[-1] if matches else ""


def is_high_value(state):
    return decision(state, "risk_analysis", "RISK_CATEGORY") == "HIGH_VALUE"


def is_suspicious(state):
    return decision(state, "risk_analysis", "RISK_CATEGORY") == "SUSPICIOUS"


def is_low_risk(state):
    return decision(state, "risk_analysis", "RISK_CATEGORY") == "LOW_RISK"


def is_in_stock(state):
    return decision(state, "inventory_check", "STOCK_STATUS") == "IN_STOCK"


def is_out_of_stock(state):
    return decision(state, "inventory_check", "STOCK_STATUS") == "OUT_OF_STOCK"


print("✓ Routing conditions defined")

✓ Routing conditions defined


In [5]:
builder = GraphBuilder()

# Nodes
builder.add_node(risk_analyzer, "risk_analysis")
builder.add_node(manual_review_agent, "manual_review")
builder.add_node(fraud_detection_agent, "fraud_check")
builder.add_node(inventory_agent, "inventory_check")
builder.add_node(shipping_agent, "shipping")
builder.add_node(refund_agent, "refund")

# Conditional edges out of risk analysis
builder.add_edge("risk_analysis", "manual_review", condition=is_high_value)
builder.add_edge("risk_analysis", "fraud_check", condition=is_suspicious)
builder.add_edge("risk_analysis", "inventory_check", condition=is_low_risk)

# Review / fraud check → inventory
builder.add_edge("manual_review", "inventory_check")
builder.add_edge("fraud_check", "inventory_check")

# Inventory → ship or refund
builder.add_edge("inventory_check", "shipping", condition=is_in_stock)
builder.add_edge("inventory_check", "refund", condition=is_out_of_stock)

builder.set_entry_point("risk_analysis")
graph = builder.build()
print("✓ Graph built")


def show_path(result):
    path = " → ".join(node.node_id for node in result.execution_order)
    print(f"Status: {result.status}\nPath:   {path}")
    last = result.execution_order[-1].node_id
    print(f"\nFinal step ({last}):\n{result.results[last].result}")

✓ Graph built


/tmp/ipykernel_2788/3216984768.py:25: UserWarning: graph has no max_node_executions or execution_timeout set; execution is unbounded
  graph = builder.build()


In [6]:
# High value → manual review → inventory → shipping
result = graph("Order #12345: $1,500 laptop from an existing customer with 3 previous orders; billing and shipping address match")
show_path(result)

Status: Status.COMPLETED
Path:   risk_analysis → manual_review → inventory_check → shipping

Final step (shipping):
### Shipment Processing Report

**Order Reference:** #12345  
**Item:** Premium Laptop ($1,500.00)  
**Inventory Status:** Verified (In Stock)  
**Security/Risk Assessment:** Low Risk (Existing customer, 3 previous orders, matching billing/shipping address)

The shipment for Order #12345 has been successfully processed in our logistics system. Because this is a high-value item ($1,500), the shipment has been upgraded to **Signature Required** to ensure secure delivery.

---

### Generated Shipping Label

```text
+-----------------------------------------------------------------------+
|  UPS GROUND                                             Qty: 1 of 1   |
|  Tracking #: 1Z9F93Y40123456784                                       |
+-----------------------------------------------------------------------+
|  FROM:                                                              

In [7]:
# Low risk → inventory → shipping
result = graph("Order #67890: $45 book from a returning customer with 50+ previous orders")
show_path(result)

Status: Status.COMPLETED
Path:   risk_analysis → inventory_check → shipping

Final step (shipping):
### Shipment Processing Report

**Order Reference:** #67890  
**Item:** Book ($45.00)  
**Inventory Status:** Verified (In Stock)  
**Customer Profile:** VIP Returning Customer (50+ previous orders)  
**Security/Risk Assessment:** Trusted Customer (No signature required; standard delivery)

Because this order is for a highly valued returning customer with over 50 successful orders, we have expedited the fulfillment processing. The package will be shipped via **USPS Ground Advantage** for fast, reliable delivery.

---

### Generated Shipping Label

```text
+-----------------------------------------------------------------------+
|  USPS GROUND ADVANTAGE                                  Qty: 1 of 1   |
|  Tracking #: 9400 1118 9956 0012 3456 78                              |
+-----------------------------------------------------------------------+
|  FROM:                                  

In [8]:
# Suspicious → fraud check → inventory → refund (limited edition is out of stock)
result = graph(
    "Order #11111: $800 limited edition headphones, account created 5 minutes ago, "
    "shipping to a different country than the billing address"
)
show_path(result)

Status: Status.COMPLETED
Path:   risk_analysis → fraud_check → inventory_check → refund

Final step (refund):
Based on the input from the inventory check confirming that the item is out of stock, I have processed the refund and initiated the payment reversal. 

Here are the details of the action taken:

### **1. Payment Reversal Details**
*   **Order ID:** #11111
*   **Refund Amount:** $800.00 USD
*   **Reason for Refund:** Item Out of Stock (Limited Edition Headphones)
*   **Status:** Refund Initiated (Reversal processing to the original payment method; funds should appear in 3-5 business days depending on the bank).
*   **Order Status Updated To:** `Cancelled - Refunded`

---

### **2. Customer Notification Sent**

**To:** [Customer Email associated with Order #11111]  
**Subject:** Important update regarding your Order #11111  

Dear Customer,

Thank you for shopping with us. 

We are writing to inform you that, unfortunately, we are unable to fulfill your order for the **Limited Ed